# Poysuwop_Nr1 / Ainu Tokenizer
Transformer based tokenizer

### 0. Library install

In [ ]:
!pip install torch accelerate transformers tokenizers huggingface sentencepiece

### 1. Preprocess

In [245]:
# Library
import os
import glob
import json
import re
import collections
import unicodedata
import pprint

# Change Current Directory
os.chdir('/content/drive/MyDrive/Colab Notebooks/Poysuwop')

#### Load dataset

In [246]:
def alphabetCheck(text):
    for char in text:
        if not (char.isascii() and (char.isalpha() or char.isspace() or '=' or "'")):
            return False
    return True

In [248]:
# Fetch sentence using the 'ain' tag from .json file
# ['ain']       romanized Ainu sentence
# ['ain-kana']  Ainu sentence transcribed in Ainu Kana
# ['jpn']       Japanese translation

texts = []

for f in glob.glob('./corpus_json/*.json'):

    # load .json
    data = json.load(open(str(f)))

    for key, value in data.items():

        if key != 'code' and key != 'title':
            sentence = value['ain']

            if len(sentence) > 0 and alphabetCheck(sentence) == True:
                texts.append(sentence)

In [249]:
##### Check charcters used in the dataset

charlist = [char for text in texts for char in list(text)]
charlist = [k for k, v in collections.Counter(charlist).items() if v > 1]

#charlist = [re.sub(r"[^a-zA-Z0-9 =]","",char) for char in charlist]
charlist_TBR = ''.join(str(x) for x in charlist)
char_TBR = re.sub(r"[a-zA-Z0-9 =]","",charlist_TBR)
print("Charcaters to be removed: {0}".format(char_TBR))

Charcaters to be removed: .'?}!_(),:"-[]`*;/@


#### Cleansing

In [250]:
# Affix cleansing

dicPrefixes = {
    'ku=':'ku= ', #1sg.nom
    'k=':'k= ', #1sg.nom
    'e=':'e= ', #2sg.nom
    'eci=':'eci= ', #2pl.nom
    'es=':'es= ', #2pl.nom [Ishikar]
    'a=':'a= ', #indefinite / inclusive 1pl.nom
    'an=':'an= ', #indefinite / inclusive 1pl.nom
    'ci=':'ci= ', #exclusive 1pl.nom
    'c=':'c= ', #exclusive 1pl.nom
    'i=':'i= ', #indefinite acc
    'en=':'en= ', #1sg.acc
    'un=':'un= ', #1pl.acc
}

dicSuffixes = {
    '=as':' =as',
    '=an':' =an',
}

def ainAffixCleanse(sentence, dic_chars, mode):
    # mode: 0 - prefix, 1 - suffix

    if mode == 0: #prefix
        for key in dic_chars.keys():
            sentence = re.sub(r'(\s|^)' + re.escape(key), r'\1' +dic_chars[key], sentence)

        # Special treatment for an=an (eng. (inclusive) we are / stay), which consists of the verb 'an' combined with the indefinite pronoun suffix '=an'
        # Both ku=ku (eng. I drink) and e=e (eng. you eat) are not subject to be addressed, as they can be correctly split into the prefix and the verb.
        dic_revise = {"an= an": "an =an"}

        for key in dic_revise.keys():
            sentence = sentence.replace(key, dic_revise[key]) if key in sentence else sentence

    else: #suffix
        for key in dic_chars.keys():
            sentence = re.sub(re.escape(key) + r'(?=\s|$)', dic_chars[key], sentence)

    sentence = ' '.join(sentence.split())

    return sentence

sentence = 'an=an'

# Clease affixes: process twice for verbs combining both a nominative and an accusative affix (e.g. eci=un=nukar)
for i in range(2):
    sentence = ainAffixCleanse(sentence, dicPrefixes, 0)
    sentence = ainAffixCleanse(sentence, dicSuffixes, 1)

sentence

'an =an'

In [251]:
# Orthography - a=kor itak
dicOrthography = {
    #'ai':'ay',
    #'au':'aw',
    #'ei':'ey',
    #'eu':'ew',
    #'iu':'iw',
    #'oi':'oy',
    #'ou':'ow',
    #'ui':'uy',
    'ch':'c',
    'sh':'s',
}

def char_cleanse(sentence, dic_chars):

    for key in dic_chars.keys():
        sentence = sentence.replace(key, dic_chars[key]) if key in sentence else sentence

    # Remove Extra Spaces
    sentence = ' '.join(sentence.split())
    return sentence

In [252]:
def preprocess(sentence):

    # Zenkaku -> Hankaku equivalent
    sentence = unicodedata.normalize('NFKD', sentence)

    # Remove number with parentheses (eg. (123) --> '', [123] --> '')
    sentence = re.sub(r'\(.*?\)|\[.*?\]', '', sentence)

    # Remove symbols
    sentence = re.sub(r"[^a-zA-Z0-9 =]","",sentence)

    # Apply orthography
    sentence = char_cleanse(sentence, dicOrthography)

    # Clease affixes: process twice for verbs combining both a nominative and an accusative affix (e.g. eci=un=nukar)
    for i in range(2):
        sentence = ainAffixCleanse(sentence, dicPrefixes, 0)
        sentence = ainAffixCleanse(sentence, dicSuffixes, 1)

    # Trim sentence
    sentence = sentence.strip()

    return sentence

# -- ex. --
sentence = 'yayán ainu ku=ne ruwe ne korka, ainu itak ani Transformer[123] ku=kor_ rushuy un!'
preprocess(sentence)

'yayan ainu ku= ne ruwe ne korka ainu itak ani Transformer ku= kor rusuy un'

In [253]:
texts[1]

'ku=wenhawehe ka an kanpi ka ta'

In [254]:
# Preprocess sentences
for i in range(len(texts)):
    texts[i] = preprocess(texts[i])

In [255]:
texts[1]

'ku= wenhawehe ka an kanpi ka ta'

In [256]:
# sample
# sentence = 'ク・ネㇷ゚キ　ヒ　カ　エン・ココパンパ　コㇿカ　ネ　ワ　アンペ　カ　ケシㇼキラㇷ゚　ワ　ク・ネㇷ゚キ　ルスイ　ワ　ケシㇼキラㇷ゚　ワ　クス　（ク・ネㇷ゚キ）　チセ　ソイ　ペカ　クネㇷ゚キ　コㇿ　カン。'
# sentence = 'ペウレクﾙ　ネ　コﾛカ　ア・アイヌコﾛ'
sentence = '“Shirokanipe ranran pishkan, konkanipe ranran pishkan.” arian rekpo chiki kane petesoro sapash aine, ainukotan enkashike chikush kor shichorpokun inkarash ko teeta wenkur tane nishpa ne, teeta nishpa tane wenkur ne kotom shiran.'
sentence_orth = preprocess(sentence)

print(sentence)
print(sentence_orth)

“Shirokanipe ranran pishkan, konkanipe ranran pishkan.” arian rekpo chiki kane petesoro sapash aine, ainukotan enkashike chikush kor shichorpokun inkarash ko teeta wenkur tane nishpa ne, teeta nishpa tane wenkur ne kotom shiran.
Shirokanipe ranran piskan konkanipe ranran piskan arian rekpo ciki kane petesoro sapas aine ainukotan enkasike cikus kor sicorpokun inkaras ko teeta wenkur tane nispa ne teeta nispa tane wenkur ne kotom siran


#### Affix Marker Check

In [257]:
# 人称接辞マーカー = の両端に文字が残っているものの抽出

cnt = 0
for i in range(len(texts)):
    if re.search(r'\S+=\S+', texts[i]):
        print(texts[i])
        cnt+=1
cnt

po hene kimoterke=a
inaw oa=asi ne ya ki kor an =an ani
earkinne arwen sakanramu aci=kore wa
pona=yupi poro a= yupi a= saha sekor hawean =an kor
aci=tekekar wa
paye=anrewsi =an pa ayne
oraun inawke=aninawroski =an kusu inawke =an kor
aci=nure hawe
ora hopuni=a hopuni =an hine
mourii=kore wa manpuri ne a= kor
onne utar tak wa asa=suke wa suke =an wa a= ipere
ari an pe aya=ye konno
ani=hotuyekar konno ahup =an wa a= tononte
ani=tura hine soy peka kim peka
sirkunpato asihikea=sike an= esiyarpokani wa
arpa =an rusuy neuna=eyaynita yakka
po hene kimoterke=a
inaw oa=asi ne ya ki kor an =an ani
arki=anpa ne hine orano
kurkotunas noyne hoastari kor apkas siri ku= nukark=erampokiwen
u=pon hi ta anakne yacipocipoci kor ku= sinot
kuneywa usey ku= kar wa ku= ku rusuy kusu a= oypep or un usey k= omare akusu kuta waku=teke cire
irukay ey=sam ta ek


23

#### Save cleansed sentences

In [258]:
# Temporary saving to a txt file
with open("./ain.txt", "w") as output:
    output.write(str(texts))

print(len(texts))

141951


### 2. Prepare Tokenizer

In [259]:
from tokenizers import (
    decoders,
    models,
    normalizers,
    pre_tokenizers,
    processors,
    trainers,
    Tokenizer
)

# Initialize a tokenizer
#tokenizer = ByteLevelBPETokenizer()
tokenizer = Tokenizer(models.WordPiece(unk_token='[UNK]'))

tokenizer.normalizer = normalizers.Sequence(
    [normalizers.Lowercase(), normalizers.NFKC()]
)

tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

In [260]:
def yield_text():
    for row in texts:
        yield row

In [261]:
trainer = trainers.WordPieceTrainer(
    vocab_size=30_000,
    special_tokens=['[UNK]', '[PAD]', '[CLS]', '[SEP]', '[MASK]'],
    min_frequency=2,
    continuing_subword_prefix='##'
)

In [262]:
tokenizer.train_from_iterator(yield_text(), trainer=trainer)

In [263]:
# Vocabulary size
tokenizer.get_vocab_size()

15669

In [264]:
cls_id = tokenizer.token_to_id('[CLS]')
sep_id = tokenizer.token_to_id('[SEP]')

In [265]:
tokenizer.post_processor = processors.TemplateProcessing(
    single=f'[CLS]:0 $A:0 [SEP]:0',
    pair=f'[CLS]:0 $A:0 [SEP]:0 $B:1 [SEP]:1',
    special_tokens=[
        ('[CLS]', cls_id),
        ('[SEP]', sep_id)
    ]
)

In [266]:
tokenizer.decoder = decoders.WordPiece(prefix='##')

In [267]:
from transformers import PreTrainedTokenizerFast

full_tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=tokenizer,
    unk_token='[UNK]',
    pad_token='[PAD]',
    cls_token='[CLS]',
    sep_token='[SEP]',
    mask_token='[MASK]'
)

#### Save tokenizer model

In [268]:
full_tokenizer.save_pretrained('ain-base-tokenizer')

('ain-base-tokenizer/tokenizer_config.json',
 'ain-base-tokenizer/special_tokens_map.json',
 'ain-base-tokenizer/tokenizer.json')

#### Check Tokenizer

In [269]:
tokenizer = PreTrainedTokenizerFast.from_pretrained('ain-base-tokenizer')

In [270]:
# Encoding - Nr1
sentence = "ohonno somo unukar=an"
print('Tokenizer results: ', tokenizer(sentence))

encodedTokens = tokenizer.encode(sentence)
print('Encoding: ', encodedTokens)

decodedTokens = tokenizer.decode(encodedTokens)
print('Decoding: ', decodedTokens)

Tokenizer results:  {'input_ids': [2, 1380, 145, 2497, 6, 53, 3], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1]}
Encoding:  [2, 1380, 145, 2497, 6, 53, 3]
Decoding:  [CLS] ohonno somo unukar = an [SEP]


In [271]:
# Encoding - Nr2
sentence = "Ku=kor wa k=an an pe hinak ta k=anu yakka k=erampewtek"
print('Tokenizer results: ', tokenizer(sentence))

encodedTokens = tokenizer.encode(sentence)
print('Encoding: ', encodedTokens)

decodedTokens = tokenizer.decode(encodedTokens)
print('Decoding: ', decodedTokens)

Tokenizer results:  {'input_ids': [2, 238, 6, 61, 59, 16, 6, 53, 53, 79, 708, 71, 16, 6, 628, 143, 16, 6, 744, 3], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}
Encoding:  [2, 238, 6, 61, 59, 16, 6, 53, 53, 79, 708, 71, 16, 6, 628, 143, 16, 6, 744, 3]
Decoding:  [CLS] ku = kor wa k = an an pe hinak ta k = anu yakka k = erampewtek [SEP]
